# Carbon-24 Feature Extraction 

## 1. Cài đặt thư viện

In [2]:
!pip install google-cloud-bigquery-storage --quiet --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.7/306.7 kB 5.5 MB/s eta 0:00:00a 0:00:01


## 2. Import

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm

from datasets import load_dataset
from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

pd.set_option('display.max_columns', None)
print('Imports OK')

Imports OK


## 3. Load dataset

In [4]:
ds = load_dataset('albertvillanova/carbon_24')
print(ds)

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/6.97M [00:00<?, ?B/s]

val.csv:   0%|          | 0.00/2.32M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/2.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6091 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2032 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2030 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'material_id', 'cif', 'energy_per_atom'],
        num_rows: 6091
    })
    validation: Dataset({
        features: ['Unnamed: 0', 'material_id', 'cif', 'energy_per_atom'],
        num_rows: 2032
    })
    test: Dataset({
        features: ['Unnamed: 0', 'material_id', 'cif', 'energy_per_atom'],
        num_rows: 2030
    })
})


In [5]:
dfs = []
for split in ds.keys():
    tmp = ds[split].to_pandas()
    tmp['split'] = split
    tmp['source_file'] = 'huggingface'
    dfs.append(tmp)

carbon_df = pd.concat(dfs, ignore_index=True)
print('carbon_df shape:', carbon_df.shape)
print('columns:', carbon_df.columns.tolist())
display(carbon_df.head(3))

carbon_df shape: (10153, 6)
columns: ['Unnamed: 0', 'material_id', 'cif', 'energy_per_atom', 'split', 'source_file']


,Unnamed: 0,material_id,cif,energy_per_atom,split,source_file
0,9251,C-130499-1826-36,# generated using pymatgen\ndata_C\n_symmetry_...,-154.311891,train,huggingface
1,6412,C-13904-4247-31,# generated using pymatgen\ndata_C\n_symmetry_...,-154.332183,train,huggingface
2,3301,C-92138-4782-35,# generated using pymatgen\ndata_C\n_symmetry_...,-154.178157,train,huggingface


## 4. Thiết lập thư mục

In [6]:
PROJECT_DIR  = '/kaggle/working/carbon24_project_v2'
DATA_DIR     = f'{PROJECT_DIR}/data'
CHECKPT_DIR  = f'{PROJECT_DIR}/checkpoints'

for d in [DATA_DIR, CHECKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Directories ready.')

Directories ready.


## 5. Xác định cột CIF và energy

In [7]:
# Tự động phát hiện cột CIF
cif_col = next(
    (col for col in carbon_df.columns
     if carbon_df[col].dropna().astype(str).head(20)
        .str.contains('_cell_length|_atom_site|data_').any()),
    None
)

# Tự động phát hiện cột energy
energy_col = next(
    (col for col in carbon_df.columns
     if any(kw in col.lower() for kw in ['energy', 'formation', 'enthalpy'])),
    None
)

print('cif_col   :', cif_col)
print('energy_col:', energy_col)

cif_col   : cif
energy_col: energy_per_atom


## 6. Kiểm tra bond_cutoff trên mẫu nhỏ

Trước khi chạy toàn bộ, kiểm tra phân phối `max_bond_length` trên 200 cấu trúc để xác nhận cutoff 2.0 Å là đủ.

In [8]:
PROBE_CUTOFF = 2.5   # cutoff rộng để thăm dò
probe_max_bonds = []

for _, row in tqdm(carbon_df.sample(200, random_state=42).iterrows(),
                   total=200, desc='Probing bond lengths'):
    try:
        st = Structure.from_str(row[cif_col], fmt='cif')
        for site in st:
            neighs = st.get_neighbors(site, PROBE_CUTOFF)
            dists = [n.nn_distance for n in neighs if n.nn_distance > 0.5]
            if dists:
                probe_max_bonds.append(max(dists))
    except Exception:
        pass

probe_series = pd.Series(probe_max_bonds)
print('Bond length distribution (probe cutoff=2.5 Å):')
print(probe_series.describe())
print(f'\n95th percentile: {probe_series.quantile(0.95):.4f} Å')
print(f'99th percentile: {probe_series.quantile(0.99):.4f} Å')
print('\n=> Nếu 99th percentile < 2.0 Å thì cutoff=2.0 là an toàn.')

Probing bond lengths: 100%|██████████| 200/200 [00:01<00:00, 184.53it/s]

Bond length distribution (probe cutoff=2.5 Å):
count    1670.000000
mean        2.463058
std         0.041529
min         2.152277
25%         2.453231
50%         2.476461
75%         2.487257
max         2.499989
dtype: float64

95th percentile: 2.4971 Å
99th percentile: 2.4998 Å

=> Nếu 99th percentile < 2.0 Å thì cutoff=2.0 là an toàn.


In [9]:
# Đặt cutoff dựa trên kết quả thăm dò
# Mặc định 2.0 Å — đủ cho C-C sp/sp2/sp3 (max ~1.54 Å) với biên an toàn
BOND_CUTOFF = 2.0
SYMPREC_STRICT = 0.01   # chính xác hơn v1 (0.1)
SYMPREC_LOOSE  = 0.1    # fallback nếu strict thất bại

print(f'BOND_CUTOFF  = {BOND_CUTOFF} Å')
print(f'SYMPREC      = {SYMPREC_STRICT} (fallback {SYMPREC_LOOSE})')

BOND_CUTOFF  = 2.0 Å
SYMPREC      = 0.01 (fallback 0.1)


## 7. Hàm trích xuất đặc trưng (v2)

### Đặc trưng mới so với v1

| Đặc trưng | Mô tả |
|---|---|
| `lattice_anisotropy` | max(a,b,c) / min(a,b,c) — đo mức độ dị hướng mạng |
| `angle_std` | std(alpha, beta, gamma) — đo biến dạng góc |
| `bond_length_range` | max - min bond length — đo độ đồng đều liên kết |
| `fraction_sp` | tỉ lệ nguyên tử có coordination=2 (sp, carbyne) |
| `fraction_sp2` | tỉ lệ nguyên tử có coordination=3 (sp², graphene) |
| `fraction_sp3` | tỉ lệ nguyên tử có coordination=4 (sp³, diamond) |
| `packing_fraction` | ước lượng mật độ đóng gói dựa trên bán kính C |
| `symprec_used` | giá trị symprec thực sự dùng (strict hay loose) |

In [10]:
# Bán kính cộng hóa trị của Carbon (Å)
C_RADIUS = 0.77

def extract_features_v2(
    cif_str,
    symprec_strict=SYMPREC_STRICT,
    symprec_loose=SYMPREC_LOOSE,
    bond_cutoff=BOND_CUTOFF,
    compute_symmetry=True,
    compute_bonding=True
):
    """
    Trích xuất đặc trưng từ chuỗi CIF của cấu trúc carbon.
    Trả về dict với đầy đủ đặc trưng hình học, đối xứng, liên kết và hybridization.
    """
    structure = Structure.from_str(cif_str, fmt='cif')
    lattice   = structure.lattice

    # ── Thông số cơ bản ──────────────────────────────────────────────────────
    num_atoms       = len(structure)
    volume          = float(structure.volume)
    volume_per_atom = volume / num_atoms if num_atoms > 0 else np.nan
    density         = float(structure.density)

    # ── Thông số mạng tinh thể ───────────────────────────────────────────────
    a     = float(lattice.a)
    b     = float(lattice.b)
    c     = float(lattice.c)
    alpha = float(lattice.alpha)
    beta  = float(lattice.beta)
    gamma = float(lattice.gamma)

    b_over_a = b / a if a != 0 else np.nan
    c_over_a = c / a if a != 0 else np.nan

    # Tổng độ lệch góc so với 90° (v1)
    angle_deviation = abs(alpha - 90) + abs(beta - 90) + abs(gamma - 90)

    # [MỚI] Std của 3 góc — đo biến dạng góc đồng đều hơn
    angle_std = float(np.std([alpha, beta, gamma]))

    # [MỚI] Dị hướng mạng — phân biệt cấu trúc layered vs isotropic
    lattice_anisotropy = max(a, b, c) / min(a, b, c) if min(a, b, c) != 0 else np.nan

    # ── Đối xứng ─────────────────────────────────────────────────────────────
    space_group_number = np.nan
    space_group_symbol = 'unknown'
    crystal_system     = 'unknown'
    symprec_used       = np.nan

    if compute_symmetry:
        # Thử strict trước, fallback về loose
        for sp in [symprec_strict, symprec_loose]:
            try:
                sga = SpacegroupAnalyzer(structure, symprec=sp)
                space_group_number = sga.get_space_group_number()
                space_group_symbol = sga.get_space_group_symbol()
                crystal_system     = sga.get_crystal_system()
                symprec_used       = sp
                break
            except Exception:
                continue

    # ── Liên kết và coordination ─────────────────────────────────────────────
    bond_lengths = []
    coord_nums   = []

    if compute_bonding:
        try:
            for site in structure:
                neighs = structure.get_neighbors(site, bond_cutoff)
                # Lọc khoảng cách hợp lý: > 0.5 Å (loại self-image)
                dists = [float(n.nn_distance) for n in neighs
                         if 0.5 < n.nn_distance <= bond_cutoff]
                coord_nums.append(len(dists))
                bond_lengths.extend(dists)
        except Exception:
            bond_lengths = []
            coord_nums   = []

    if bond_lengths:
        mean_bond_length  = float(np.mean(bond_lengths))
        std_bond_length   = float(np.std(bond_lengths))
        min_bond_length   = float(np.min(bond_lengths))
        max_bond_length   = float(np.max(bond_lengths))
        # [MỚI] Khoảng biến thiên độ dài liên kết
        bond_length_range = max_bond_length - min_bond_length
    else:
        mean_bond_length  = np.nan
        std_bond_length   = np.nan
        min_bond_length   = np.nan
        max_bond_length   = np.nan
        bond_length_range = np.nan

    if coord_nums:
        mean_coordination = float(np.mean(coord_nums))
        std_coordination  = float(np.std(coord_nums))
        min_coordination  = float(np.min(coord_nums))
        max_coordination  = float(np.max(coord_nums))

        # [MỚI] Phân loại hybridization theo coordination number
        # sp  (linear, carbyne)  : coord = 2
        # sp2 (graphene, fullerene): coord = 3
        # sp3 (diamond, lonsdaleite): coord = 4
        n_total  = len(coord_nums)
        n_sp     = sum(1 for c in coord_nums if c == 2)
        n_sp2    = sum(1 for c in coord_nums if c == 3)
        n_sp3    = sum(1 for c in coord_nums if c == 4)
        n_other  = n_total - n_sp - n_sp2 - n_sp3

        fraction_sp    = n_sp    / n_total
        fraction_sp2   = n_sp2   / n_total
        fraction_sp3   = n_sp3   / n_total
        fraction_other = n_other / n_total
    else:
        mean_coordination = np.nan
        std_coordination  = np.nan
        min_coordination  = np.nan
        max_coordination  = np.nan
        fraction_sp       = np.nan
        fraction_sp2      = np.nan
        fraction_sp3      = np.nan
        fraction_other    = np.nan

    # [MỚI] Packing fraction: thể tích cầu C / thể tích ô mạng
    # Dùng bán kính cộng hóa trị C = 0.77 Å
    vol_atoms       = num_atoms * (4/3) * np.pi * (C_RADIUS ** 3)
    packing_fraction = vol_atoms / volume if volume > 0 else np.nan

    formula  = structure.composition.reduced_formula
    elements = [str(el) for el in structure.composition.elements]

    return {
        # Định danh
        'formula'  : formula,
        'elements' : ','.join(elements),
        'num_atoms': int(num_atoms),

        # Thông số mạng
        'a'    : a,
        'b'    : b,
        'c'    : c,
        'alpha': alpha,
        'beta' : beta,
        'gamma': gamma,
        'volume'         : volume,
        'volume_per_atom': float(volume_per_atom),
        'density'        : density,

        # Tỉ lệ và biến dạng mạng
        'b_over_a'          : float(b_over_a),
        'c_over_a'          : float(c_over_a),
        'angle_deviation'   : float(angle_deviation),
        'angle_std'         : angle_std,           # [MỚI]
        'lattice_anisotropy': float(lattice_anisotropy),  # [MỚI]

        # Đối xứng
        'space_group_number': space_group_number,
        'space_group_symbol': space_group_symbol,
        'crystal_system'    : crystal_system,
        'symprec_used'      : symprec_used,        # [MỚI]

        # Liên kết
        'mean_bond_length' : mean_bond_length,
        'std_bond_length'  : std_bond_length,
        'min_bond_length'  : min_bond_length,
        'max_bond_length'  : max_bond_length,
        'bond_length_range': bond_length_range,    # [MỚI]

        # Coordination
        'mean_coordination': mean_coordination,
        'std_coordination' : std_coordination,
        'min_coordination' : min_coordination,
        'max_coordination' : max_coordination,

        # Hybridization [MỚI]
        'fraction_sp'   : fraction_sp,
        'fraction_sp2'  : fraction_sp2,
        'fraction_sp3'  : fraction_sp3,
        'fraction_other': fraction_other,

        # Packing [MỚI]
        'packing_fraction': float(packing_fraction),
    }

## 8. Kiểm tra hàm trên 5 mẫu

In [11]:
test_rows = []
for idx, row in carbon_df.head(5).iterrows():
    feat = extract_features_v2(row[cif_col])
    feat['row_index']   = idx
    feat['split']       = row['split']
    feat['material_id'] = row.get('material_id', np.nan)
    feat['energy']      = row[energy_col] if energy_col else np.nan
    test_rows.append(feat)

test_df = pd.DataFrame(test_rows)
print('Columns:', test_df.columns.tolist())
display(test_df[[
    'material_id', 'num_atoms', 'density', 'crystal_system',
    'lattice_anisotropy', 'angle_std', 'bond_length_range',
    'fraction_sp', 'fraction_sp2', 'fraction_sp3',
    'packing_fraction', 'symprec_used'
]])

Columns: ['formula', 'elements', 'num_atoms', 'a', 'b', 'c', 'alpha', 'beta', 'gamma', 'volume', 'volume_per_atom', 'density', 'b_over_a', 'c_over_a', 'angle_deviation', 'angle_std', 'lattice_anisotropy', 'space_group_number', 'space_group_symbol', 'crystal_system', 'symprec_used', 'mean_bond_length', 'std_bond_length', 'min_bond_length', 'max_bond_length', 'bond_length_range', 'mean_coordination', 'std_coordination', 'min_coordination', 'max_coordination', 'fraction_sp', 'fraction_sp2', 'fraction_sp3', 'fraction_other', 'packing_fraction', 'row_index', 'split', 'material_id', 'energy']


,material_id,num_atoms,density,crystal_system,lattice_anisotropy,angle_std,bond_length_range,fraction_sp,fraction_sp2,fraction_sp3,packing_fraction,symprec_used
0,C-130499-1826-36,6,3.419546,monoclinic,2.052207,7.426418,0.077492,0.0,0.0,1.0,0.327878,0.01
1,C-13904-4247-31,10,3.533540,monoclinic,2.388054,13.349208,0.158813,0.0,0.0,1.0,0.338808,0.01
2,C-92138-4782-35,10,3.411869,triclinic,2.440570,18.148416,0.090456,0.0,0.2,0.8,0.327142,0.01
3,C-192672-505-73,20,3.588599,triclinic,4.518366,9.461977,0.166575,0.0,0.0,1.0,0.344087,0.01
4,C-193956-5355-22,6,2.760571,triclinic,1.419670,21.112942,0.068063,0.0,1.0,0.0,0.264693,0.01


## 9. Trích xuất toàn bộ dữ liệu

In [12]:
feature_rows = []
failed_rows  = []

CHECKPOINT_EVERY = 500
partial_path = f'{CHECKPT_DIR}/carbon24_features_v2_partial.csv'
failed_path  = f'{CHECKPT_DIR}/carbon24_failed_rows_v2.csv'

for idx, row in tqdm(carbon_df.iterrows(), total=len(carbon_df), desc='Extracting'):
    try:
        feat = extract_features_v2(row[cif_col])

        feat['row_index']   = idx
        feat['split']       = row.get('split', 'unknown')
        feat['source_file'] = row.get('source_file', 'unknown')

        for id_col in ['material_id', 'id', 'mp_id', 'jid', 'name']:
            if id_col in carbon_df.columns:
                feat[id_col] = row[id_col]

        if energy_col and energy_col in carbon_df.columns:
            feat['energy'] = row[energy_col]

        feature_rows.append(feat)

    except Exception as e:
        failed_rows.append({'row_index': idx, 'error': str(e)})

    if (idx + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(feature_rows).to_csv(partial_path, index=False)
        pd.DataFrame(failed_rows).to_csv(failed_path, index=False)
        print(f'  Checkpoint {idx+1}: ok={len(feature_rows)}, failed={len(failed_rows)}')

feat_df   = pd.DataFrame(feature_rows)
failed_df = pd.DataFrame(failed_rows)

print(f'\nDone. Features: {feat_df.shape}, Failed: {failed_df.shape}')
display(feat_df.head(3))

Extracting:   2%|▏         | 184/10153 [00:01<01:19, 124.82it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:   5%|▌         | 514/10153 [00:03<01:15, 127.03it/s]

  Checkpoint 500: ok=500, failed=0


Extracting:   9%|▊         | 865/10153 [00:06<01:12, 127.72it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:  10%|█         | 1024/10153 [00:07<01:19, 115.30it/s]

  Checkpoint 1000: ok=1000, failed=0


Extracting:  11%|█         | 1103/10153 [00:08<01:14, 121.22it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:  11%|█▏        | 1157/10153 [00:09<01:11, 126.70it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:  15%|█▍        | 1514/10153 [00:12<01:27, 98.92it/s] 

...

Extracting:  94%|█████████▍| 9591/10153 [01:20<00:05, 111.32it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:  96%|█████████▋| 9786/10153 [01:22<00:02, 128.17it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:  97%|█████████▋| 9800/10153 [01:22<00:02, 129.37it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:  98%|█████████▊| 9985/10153 [01:23<00:01, 131.25it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:  99%|█████████▊| 10026/10153 [01:24<00:02, 63.44it/s]

  Checkpoint 10000: ok=10000, failed=0


Extracting:  99%|█████████▉| 10067/10153 [01:24<00:00, 97.12it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting:  99%|█████████▉| 10081/10153 [01:24<00:00, 105.74it/s]spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
Extracting: 100%|██████████| 10153/10153 [01:25<00:00, 118.72it/s]



Done. Features: (10153, 40), Failed: (0, 0)


,formula,elements,num_atoms,a,b,c,alpha,beta,gamma,volume,volume_per_atom,density,b_over_a,c_over_a,angle_deviation,angle_std,lattice_anisotropy,space_group_number,space_group_symbol,crystal_system,symprec_used,mean_bond_length,std_bond_length,min_bond_length,max_bond_length,bond_length_range,mean_coordination,std_coordination,min_coordination,max_coordination,fraction_sp,fraction_sp2,fraction_sp3,fraction_other,packing_fraction,row_index,split,source_file,material_id,energy
0,C,C,6,2.48053,5.09056,4.89497,57.97521,59.51109,43.04559,34.994539,5.832423,3.419546,2.052207,1.973357,109.46811,7.426418,2.052207,12,C2/m,monoclinic,0.01,1.526933,0.023654,1.491934,1.569426,0.077492,4.0,0.0,4.0,4.0,0.0,0.0,1.0,0.0,0.327878,0,train,huggingface,C-130499-1826-36,-154.311891
1,C,C,10,2.48986,4.65679,5.94592,90.03290,77.93642,57.67568,56.442646,5.644265,3.533540,1.870302,2.388054,44.42080,13.349208,2.388054,12,C2/m,monoclinic,0.01,1.529398,0.042382,1.433922,1.592736,0.158813,4.0,0.0,4.0,4.0,0.0,0.0,1.0,0.0,0.338808,1,train,huggingface,C-13904-4247-31,-154.332183
2,C,C,10,2.44168,4.79849,5.95909,119.73602,101.56104,75.51457,58.455462,5.845546,3.411869,1.965241,2.440570,55.78249,18.148416,2.440570,1,P1,triclinic,0.01,1.512933,0.027126,1.462408,1.552865,0.090456,3.8,0.4,3.0,4.0,0.0,0.2,0.8,0.0,0.327142,2,train,huggingface,C-92138-4782-35,-154.178157


## 10. Tính `relative_energy` — chỉ dùng min của tập train

> **Lý do:** Tính `min()` trên toàn bộ dataset (gộp train+val+test) gây **data leakage** khi dùng cho bài toán dự đoán năng lượng. Phiên bản này tính baseline từ tập train rồi áp dụng cho tất cả.

In [13]:
if 'energy' in feat_df.columns:
    feat_df['energy'] = pd.to_numeric(feat_df['energy'], errors='coerce')

    # Baseline = min energy của tập TRAIN
    train_min_energy = feat_df.loc[feat_df['split'] == 'train', 'energy'].min()
    print(f'Train min energy : {train_min_energy:.6f} eV/atom')
    print(f'Global min energy: {feat_df["energy"].min():.6f} eV/atom')

    feat_df['relative_energy'] = feat_df['energy'] - train_min_energy

    print('\nRelative energy stats:')
    print(feat_df.groupby('split')['relative_energy'].describe().round(4))
else:
    print('Không tìm thấy cột energy.')

Train min energy : -154.556275 eV/atom
Global min energy: -154.556985 eV/atom

Relative energy stats:
             count    mean     std     min     25%     50%     75%     max
split                                                                     
test        2030.0  0.3061  0.1364 -0.0007  0.2240  0.3398  0.4208  0.4905
train       6091.0  0.3052  0.1374  0.0000  0.2229  0.3375  0.4193  0.4907
validation  2032.0  0.3070  0.1339  0.0022  0.2213  0.3415  0.4178  0.4902


## 11. Kiểm tra chất lượng đặc trưng

In [14]:
# Missing values
missing = feat_df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if missing.empty:
    print('Không có missing values.')
else:
    print('Missing values:')
    print(missing)
    print()
    missing_report = missing.reset_index()
    missing_report.columns = ['column', 'missing_count']
    missing_report['missing_ratio'] = missing_report['missing_count'] / len(feat_df)
    display(missing_report)

Không có missing values.


In [15]:
# Phân phối hybridization
print('=== Phân phối hybridization (fraction trung bình theo split) ===')
hybrid_cols = ['fraction_sp', 'fraction_sp2', 'fraction_sp3', 'fraction_other']
print(feat_df.groupby('split')[hybrid_cols].mean().round(4))

print('\n=== Phân phối hybridization (toàn bộ dataset) ===')
print(feat_df[hybrid_cols].describe().round(4))

=== Phân phối hybridization (fraction trung bình theo split) ===
            fraction_sp  fraction_sp2  fraction_sp3  fraction_other
split                                                              
test             0.0003        0.3957        0.6038          0.0001
train            0.0005        0.4022        0.5971          0.0003
validation       0.0005        0.4032        0.5962          0.0002

=== Phân phối hybridization (toàn bộ dataset) ===
       fraction_sp  fraction_sp2  fraction_sp3  fraction_other
count   10153.0000    10153.0000    10153.0000      10153.0000
mean        0.0004        0.4011        0.5982          0.0002
std         0.0081        0.3900        0.3902          0.0058
min         0.0000        0.0000        0.0000          0.0000
25%         0.0000        0.0000        0.2500          0.0000
50%         0.0000        0.3333        0.6667          0.0000
75%         0.0000        0.7500        1.0000          0.0000
max         0.2000        1.0000        

In [16]:
# Kiểm tra symprec_used — bao nhiêu cấu trúc cần fallback về loose?
if 'symprec_used' in feat_df.columns:
    sp_counts = feat_df['symprec_used'].value_counts(dropna=False)
    print('symprec_used distribution:')
    print(sp_counts)
    pct_loose = (feat_df['symprec_used'] == 0.1).sum() / len(feat_df) * 100
    print(f'\n{pct_loose:.1f}% cấu trúc cần fallback symprec=0.1')

symprec_used distribution:
symprec_used
0.01    10153
Name: count, dtype: int64

0.0% cấu trúc cần fallback symprec=0.1


In [17]:
# Thống kê tổng quan
print('=== Thống kê đặc trưng số ===')
numeric_cols = feat_df.select_dtypes(include=np.number).columns.tolist()
display(feat_df[numeric_cols].describe().T.round(4))

=== Thống kê đặc trưng số ===


,count,mean,std,min,25%,50%,75%,max
num_atoms,10153.0,9.2109,3.5817,6.0000,6.0000,8.0000,10.0000,24.0000
a,10153.0,2.7811,0.5691,2.4008,2.4566,2.4831,3.0419,6.5080
b,10153.0,4.2905,1.0831,2.4000,3.6132,4.2023,4.8069,10.7948
c,10153.0,5.8114,1.6049,2.4629,4.6501,5.6158,6.5799,17.3164
alpha,10153.0,89.0238,19.9037,30.2113,74.8064,89.9471,104.4288,146.6627
beta,10153.0,89.6800,15.5195,35.6290,78.6830,89.9973,100.8968,140.7371
gamma,10153.0,89.1430,17.3075,33.2787,75.4481,89.9947,101.5604,151.6675
volume,10153.0,58.3758,22.6450,32.5338,43.0936,52.4713,69.0137,175.0343
volume_per_atom,10153.0,6.3682,0.6715,5.4223,5.8171,6.1376,7.0970,9.0519
density,10153.0,3.1659,0.3240,2.2033,2.8102,3.2495,3.4286,3.6782


## 12. Lưu file

In [18]:
features_path      = f'{DATA_DIR}/carbon24_features_v2.csv'
failed_final_path  = f'{DATA_DIR}/carbon24_failed_rows_v2.csv'
missing_report_path = f'{DATA_DIR}/carbon24_missing_report_v2.csv'

feat_df.to_csv(features_path, index=False)
failed_df.to_csv(failed_final_path, index=False)

# Lưu missing report
missing_full = feat_df.isna().sum().reset_index()
missing_full.columns = ['column', 'missing_count']
missing_full['missing_ratio'] = missing_full['missing_count'] / len(feat_df)
missing_full.to_csv(missing_report_path, index=False)

print(f'Features saved  : {features_path}')
print(f'Failed rows     : {failed_final_path}')
print(f'Missing report  : {missing_report_path}')
print(f'\nShape: {feat_df.shape}')
print(f'Columns ({len(feat_df.columns)}): {feat_df.columns.tolist()}')

Features saved  : /kaggle/working/carbon24_project_v2/data/carbon24_features_v2.csv
Failed rows     : /kaggle/working/carbon24_project_v2/data/carbon24_failed_rows_v2.csv
Missing report  : /kaggle/working/carbon24_project_v2/data/carbon24_missing_report_v2.csv

Shape: (10153, 41)
Columns (41): ['formula', 'elements', 'num_atoms', 'a', 'b', 'c', 'alpha', 'beta', 'gamma', 'volume', 'volume_per_atom', 'density', 'b_over_a', 'c_over_a', 'angle_deviation', 'angle_std', 'lattice_anisotropy', 'space_group_number', 'space_group_symbol', 'crystal_system', 'symprec_used', 'mean_bond_length', 'std_bond_length', 'min_bond_length', 'max_bond_length', 'bond_length_range', 'mean_coordination', 'std_coordination', 'min_coordination', 'max_coordination', 'fraction_sp', 'fraction_sp2', 'fraction_sp3', 'fraction_other', 'packing_fraction', 'row_index', 'split', 'source_file', 'material_id', 'energy', 'relative_energy']


In [19]:
%cd /kaggle/working

!zip -r carbon24_features_v2.zip \
    carbon24_project_v2/data/carbon24_features_v2.csv \
    carbon24_project_v2/data/carbon24_failed_rows_v2.csv \
    carbon24_project_v2/data/carbon24_missing_report_v2.csv

!ls -lh carbon24_features_v2.zip

/kaggle/working
  adding: carbon24_project_v2/data/carbon24_features_v2.csv (deflated 62%)
  adding: carbon24_project_v2/data/carbon24_failed_rows_v2.csv (stored 0%)
  adding: carbon24_project_v2/data/carbon24_missing_report_v2.csv (deflated 64%)
-rw-r--r-- 1 root root 1.8M May 20 05:16 carbon24_features_v2.zip
